<a href="https://colab.research.google.com/github/stacy-kendi/Binary_Representation_GA/blob/main/Genetic_Algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Genetic Algorithms What They Are and How they Work

#Solution
The answer to the problem being solved

eg in Knapsack Problem the answer is the number of items and the items to fit in the bag according to the required weight of the bag

or the Travel Salesman Problem (TSP) the solution is the actual list in the order the person is to travel to visit all cities in the shortest time possible

#Genetic Representation
How this solution is translated into a genome or a list of strings and numbers that the computer can process

eg the 5 cities for TSP could be a list or array of numbers eg [1,2,3,4,5]. The whole list is the chromosome, a single city number is the gene eg city 1 is a gene, city 2 etc (Individual elements = gene)


#Types of Genetic Representations
1. Binary -
Uses 0s and 1s
like Knapsack where you make a binary decision either yes or no

2. Permutation Representation -
Uses numbers where the order matters.
Used when you want to find the best order, schedule or sequence to a fixed set of items
eg the Travelling Salesman Problem

3. Value Represention -
Uses numbers, letters or words
where the solution requires specific measurements or continuous numbers or prices that aren't binary.
Something like a regression problem where the specific solution is important eg airplane wing dimensions need to be specific for a smooth flight or Nasa spaceship etc

4. Tree Representation -
Uses a hierarchy of functions and values
where you need to make changes to a program code/software, a mathematical formula or a set of rules
eg creating synthetic neighbours to predictions made by a model


#What is Needed
1. The genetic representation
2. Rules/constraints to prevent the Genetic Algorithm from making illegal or unwanted moves
3. Operators to determine how slicing for the cross-over and how mutation will be done


In [ ]:
#Solution - The answer to the problem being solved eg in Knapsack Problem the answer is the number of items and the items to fit in the bag according to the required weight of the bag
            # or the Travel Salesman Problem (TSP) the solution is the actual list in the order the person is to travel to visit all cities in the shortest time possible
#Genetic Representation - How this solution is translated into a genome or a list of strings and numbers that the computer can process eg the 5 cities for TSP could be a list or array of numbers eg [1,2,3,4,5]. The whole list is the chromosome, a single city number is the gene eg city 1 is a gene, city 2 etc (Individual elements = gene)

#Types of Genetic Representations
#1. Binary -
#Uses 0s and 1s
#like Knapsack where you make a binary decision either yes or no

#2. Permutation Representation -
#Uses numbers where the order matters. eg delivery route sequence
#Used when you want to find the best order, schedule or sequence to a fixed set of items
#eg the Travelling Salesman Problem

#3. Value Represention -
#Uses numbers, letters or words eg airplane wing dimensions or neural network weights
#where the solution requires specific measurements or continuous numbers or prices that aren't binary.
#Something like a regression problem where the specific solution is important eg airplane wing dimensions need to be specific for a smooth flight or Nasa spaceship etc

#4. Tree Representation -
#Uses a hierarchy of functions and values
#where you need to make changes to a program code/software, a mathematical formula or a set of rules
#eg creating synthetic neighbours to predictions made by a model

#What is Needed
#1. The genetic representation
#2. Rules/constraints to prevent the Genetic Algorithm from making illegal or unwanted moves
#3. Operators to determine how slicing for the cross-over and how mutation will be done


In [14]:
#Genetic Algorithm Implementation to Understand its Practical Implementation for Use in Other Projects
#Code is Adapted and Based on the Tutorial By - https://github.com/kiecodes/genetic-algorithms/tree/master
#Tutorial - https://www.youtube.com/watch?v=nhT56blfRpE

from collections import namedtuple
from random import choices, randint, randrange
from typing import Callable, List, Optional, Tuple
from functools import partial
import random

#1. A Genetic Algorithm Begins by Generating the Genetic Representations Randomly ie it will generate a random list of numbers to represent the cities we need for a solution or items to pack equals to the length of the solution we need

#Genetic Representation of a Solution
Genome = List[int] #single genetic representation
Population = List[Genome] #population of multiple genetic representations
#A Thing is a structure with a name, value and weight ie Each Individual Item to Pack
Thing = namedtuple('Thing', ['name', 'value', 'weight'])
FitnessFunction = Callable[[Genome], int] #implements the fitnmess to get the weights of the solutions at the selection level
PopulateFunction = Callable[[], Population] #the function takes nothing and outputs a population
SelectionFunction = Callable[[Population, FitnessFunction], Tuple[Genome, Genome]] #pairwise solution selection to output two genomes to crossover and mutate
CrossoverFunction = Callable[[Genome, Genome], Tuple[Genome, Genome]]
MutationFunction = Callable[[Genome], Genome] #takes one genome and outputs the modified genome that has been changed

#The functions separate the problem to be solved and the genetic algorithm functions are used as parameters

first_example = [
    Thing('Laptop', 500, 2200),
    Thing('Headphones', 150, 160),
    Thing('Coffee Mug', 60, 350),
    Thing('Notepad', 40, 333),
    Thing('Water Bottle', 30, 192),
]

second_example = [
    Thing('Mints', 5, 25),
    Thing('Socks', 10, 38),
    Thing('Tissues', 15, 80),
    Thing('Phone', 500, 200),
    Thing('Baseball Cap', 100, 70)
] + first_example

def generate_genome(length: int) -> Genome:
    return [random.randint(0, 1) for _ in range(length)]
    #return choices([0,1], k=length)

    #a random list of 1s and 0s of a specified length where the length = the length of the items to choose from to fit in the bag

#2. Decodes the genetic representation to see the solution it represents ie it uses a function to generate the solutions

#Function to Generate A New Solution - This creates a population of solutions several solutions to check in parallel
#Creating the Genome Multiple Times until our population gets to a desired size
def generate_population(size: int, genome_length: int) -> Population:
    return [generate_genome(genome_length) for _ in range(size)]


#3. Evaluating and Grading the Solution Generated Using a Fitness Function - Higher Fitness = Better Solution

#Fitness Function to Evaluate and Grade the Solution
#Takes in the genome representation, the list of things to choose from ie for knapsack the items to pack in the bag and weight and the input parameters to return the fitness values
FitnessFunction = Callable[[Genome], int]

def fitness(genome: Genome, things: List[Thing], weight_limit: int) -> int:
  #checking if the genome representation equals the length of the things to select from
  if len(genome) != len(things):
    raise ValueError("genome and things must be of the same length")

  weight = 0
  value = 0

  #Looping over all items/things to fit in the bag to see if it is part of the solution
  #This is by checking if the genome has a 1 at a given index
  #then add the value and weight to the accumulation variables of weights and values
  #if the total weight in the weight variable weight exceeds the weight limit , we abort the iteration and return fitness of 0 since that solution is invalid
  #if we get the end of the list without exceeding the limit we return the accumulated value

  for i, thing in enumerate(things):
    if genome[i] == 1: #if the representation at the specified index in the things ==1, ie if first item to pack ie a book =1 then add the weight and value. Each item has an index in the things
      weight += thing.weight
      value += thing.value

      if weight > weight_limit:
        return 0

  return value

      #


#4. Selection Function to Select Solutions to Generate the Next Generation
#Select a pair of solutions from the populations to be the parents of the next generation. The solutions with a higher fitness are more likely to be chosen
def selection_pair(population: Population, fitness_function: FitnessFunction) -> Population:
    return choices(
        population=population,
        weights=[fitness_function(genome) for genome in population], #handover weights for each element it can choose from. For each list of solutions in the population, get the fitness score weight to ensure the fittest solution is what is chosen
        k=2 #draw twice to get a pair
    )

#We don't call our fitness function defined above directly because the fitness function is tailored according to the problem being solved
#While the Fitness funtion in use by the selection function only needs a genome and returns a fitness value to make a choice, we use Callable function to get the genome and pass the output we want)


#5. Mixing the Best Solutions Together from the Evaluation Based on the Solution with the Best Grade

#Cross-Over Function to Mix the Best Solutions
#Takes two genomes as parameters and returns two genomes as outputs
#Select an index to cut the genomes in half for the crossover
def single_point_crossover(a: Genome, b: Genome) -> Tuple[Genome, Genome]:
  if len(a) != len(b): #checks if the two genomes from the population are of the same length
    raise ValueError("Genomes a and b must be of the same length")

  length = len(a)

  if length < 2: #a check to ensure length of any of the genomes is 2 or more to be able to split into half (a length of 1 cannot be split)
    return a, b

  p = randint(1, length -1) #selecting the index any random number from 1 to the end of the length of the genome to cut the genome in half

  return a[0:p] + b[p:], b[0:p] + a[p:] #takes the first half of the first genome and the second half of the genome b and put together to get the solution. then second solution is from the opposite half's

#6. Making Random Changes to the Mixed Solutions to Generate New Better Solutions

#Mutation Function to Make Random Changes to the Mixed/Cross-Over Solutions
#Takes a genome and using a probability changes the values ie 1's to 0's and vice versa at random positions
#Choose a random index
#If the index is higher than the probability we dont change, if less its changed with the probability
#absolute value of the current value -1 ,
def mutation(genome: Genome, num: int = 1, probability: float = 0.5) -> Genome:
  for _ in range(num):
    index = randrange(len(genome)) #choosing a random index
    genome[index] = genome[index] if random.random() > probability else abs(genome[index] - 1)
  return genome


#Evolution Loop Process
#Pass in the Parameters, the Callable Functions Defined are Substituted with the Defined Functions Which Take int the Specified Inputs and Generate The Specified Output of the Functions
#fitness limit defines a fitness condition if the fitness of the best solution exceeds the limit then the solution has been found
#Generation Limit is the Number of Generations to Run Before Ending If the Solution Doesn't Get to the Fitness Limit
def run_evolution(populate_function: PopulateFunction, fitness_function: FitnessFunction, fitness_limit: int, selection_function: SelectionFunction = selection_pair, crossover_function: CrossoverFunction = single_point_crossover, mutation_function: MutationFunction = mutation, generation_limit: int = 100) -> Tuple[Population, int]:


  population = populate_function() #Getting the First Generation

  for i in range(generation_limit): #Loop through the generation limit
                  #Sorting the population by fitness to ensure the solutions are getting the first indices of the genomes and easily keep track the fitness of the best genomes
                  #Sorting helps with elitism to be able to easily select the top genomes since they are already sorted in the population
                  population = sorted(population, key=lambda genome:fitness_function(genome), reverse=True)

                  if fitness_function(population[0]) >= fitness_limit: #Checking if the fitness limit has been reached
                    break

                  next_generation = population[0:2] #Elitism - Saving the top best solutions from one generation to move directly to the next generation before getting the other new solutions for the population, Here we keep the top 2 solutions

                  #Generating Other Solutions for the Next Generation
                  #Loop for half the length of the generation to get many solutions in the next generation as before as before and since we copied the two top solutions we can remove one loop
                  for j in range(int(len(population) / 2) - 1):
                    parents = selection_function(population, fitness_function) #getting the parents
                    offspring_a, offspring_b = crossover_function(parents[0], parents[1]) #getting the offspring
                    offspring_a = mutation_function(offspring_a) #mutating the offspring
                    offspring_b = mutation_function(offspring_b) #mutating the offspring
                    next_generation += [offspring_a, offspring_b] #adding the offspring


                  population = next_generation #Replacing the current population with the new generation

                  print(f"Generation {i}")
                  print(f"Current best solution: {max([(fitness_function(n), n) for n in population])}")

  return population, i #returning the population and the number of generations that ran
  #The i distinguishes if the algorithm went through the generation limit or if it got the best solution that fits the fitness limit early


things = first_example # Defining 'things' before calling run_evolution

#Partial used to specify the parameters to build the genome as the callable populate function takes no parameter, same case for the fitness function which takes in different parameters from the callable function
population, generations = run_evolution(
                                        populate_function=partial(generate_population, size=10, genome_length=len(things)),
                                        fitness_function=partial(fitness, things=things, weight_limit=3000),
                                        fitness_limit=740,
                                        generation_limit=100)

print(f"Number of generations: {generations}")
print(f"Best solution: {max([(fitness(n, things, 3000), n) for n in population])}")

#Translating the genetic representation back to the list of objects
def from_genome(genome: Genome, things: [Thing]) -> [Thing]:
    result = []
    for i, thing in enumerate(things):
        if genome[i] == 1: #for each thing, in the list of things, if the value of the thing at the genome =1, meaning it is part of the solution, the actual thing is added to the result
            result += [thing]

    return result #the name of the actual things which were part of the genome

print(f"Best solution: {from_genome(population[0], things)}") #genome at position 0 in the population since its sorted, meaning it is the best solution

Generation 0
Current best solution: (720, [1, 1, 0, 1, 1])
Generation 1
Current best solution: (720, [1, 1, 0, 1, 1])
Generation 2
Current best solution: (720, [1, 1, 0, 1, 1])
Generation 3
Current best solution: (740, [1, 1, 1, 0, 1])
Number of generations: 4
Best solution: (740, [1, 1, 1, 0, 1])
Best solution: [Thing(name='Laptop', value=500, weight=2200), Thing(name='Headphones', value=150, weight=160), Thing(name='Coffee Mug', value=60, weight=350), Thing(name='Water Bottle', value=30, weight=192)]
